## 項目情報（修正前）

患者番号  
性別 (1: 男性, 2: 女性)  
年齢  
入院日  
検査日  
入院-検査期間(日)  
採取状況 (1: 入院時, 2: 病棟定期)  
VRE結果 (1: 陽性, 2: 陰性)  
過去のVRE検出歴 (1: あり, 2: なし)  
感染経路 (1: もちこみ（過去歴なし）, 2: もちこみ（過去歴あり）, 3: 院内新規感染, 4: 該当なし)  
入院時申込病名  
過去3ヶ月以内の当院入院歴 (0: 該当なし, 1: あり, 2: なし)  
過去3ヶ月以内の他院入院歴 (0: 該当なし, 1: あり, 2: なし)  
入院歴のある病院 (0: 該当なし, 1: 流行地域病院の入院歴あり, 2: 入院歴なし, 3: 非流行地域の入院歴あり)  
過去3ヶ月以内の介護施設入所歴 (0: 該当なし, 1: あり, 2: なし)  
尿Ba (1: あり, 2: なし)  
個室トイレ (1: あり, 2: なし)  
おむつ使用 (1: あり, 2: なし)  
過去3ヶ月以内の抗菌薬使用歴 (1: あり, 2: なし)  
経管栄養 (1: あり, 2: なし)  

In [ ]:
import os
import pandas as pd

# --- src ---
from pathlib import Path
import sys
sys.path.append(str(Path("../").resolve()))
from src.plotting import plot_roc_curve_cv, plot_shap_summary, plot_model_roc_comparison

# --- Raw Data ---
raw_data_path = "../data/raw/"

# --- Folder to save plots ---
results_path = "../results/model_comparison/"

if not os.path.exists(results_path):
    os.makedirs(results_path)



📌 1. 基本情報（そのまま利用）
| 修正前の項目名    | 修正後の扱い                 |
| ---------- | ---------------------- |
| 患者番号       | そのまま（ID扱い、学習には使わない）    |
| 年齢         | そのまま連続値                |
| 入院日        | 入院-検査期間を使用するため学習では使わない |
| 検査日        | 同上                     |
| 入院-検査期間(日) | そのまま使用                 |

📌 2. 二値化（1/0 に変換する項目）

臨床的には あり＝1 / なし＝0 に統一して SHAP の解釈性を最大化。

| 修正前の項目名                              | 修正後（変換内容）                   |
| ------------------------------------ | --------------------------- |
| 性別 (1: 男, 2: 女)                      | 男＝1 / 女＝0                   |
| 過去のVRE検出歴 (1:あり, 2:なし)               | あり＝1 / なし＝0                 |
| 過去3ヶ月以内の当院入院歴 (0:該当なし, 1:あり, 2:なし)   | **1=1、その他=0**（該当なし/なしは同じ扱い） |
| 過去3ヶ月以内の他院入院歴 (0:該当なし, 1:あり, 2:なし)   | 同上：あり=1、その他=0               |
| 過去3ヶ月以内の介護施設入所歴 (0:該当なし, 1:あり, 2:なし) | あり=1、その他=0                  |
| 尿Ba (1:あり, 2:なし)                     | あり=1 / なし=0                 |
| 個室トイレ (1:あり, 2:なし)                   | あり=1 / なし=0                 |
| おむつ使用 (1:あり, 2:なし)                   | あり=1 / なし=0                 |
| 過去3ヶ月以内の抗菌薬使用歴 (1:あり, 2:なし)          | あり=1 / なし=0                 |
| 経管栄養 (1:あり, 2:なし)                    | あり=1 / なし=0                 |

📌 3. 多カテゴリ → One-Hot Encoding にする項目

SHAP で間違った順序関係を作らないため、カテゴリは One-Hot 化。

| 修正前の項目名                                            | 修正後の扱い                                    |
| -------------------------------------------------- | ----------------------------------------- |
| 採取状況 (1:入院時, 2:病棟定期)                               | 0/1でも良いが **One-Hot** が最も解釈しやすい            |
| 感染経路 (1:もちこみ(過去歴なし), 2:もちこみ(あり), 3:院内新規感染, 4:該当なし) | **One-Hot に変換（4カテゴリ → 4列）**               |
| 入院時申込病名                                            | テキスト → カテゴリ → One-Hot または Target Encoding |
| 入院歴のある病院 (0〜3)                                     | **One-Hot（0,1,2,3 それぞれ別列へ）**              |

📌 4. 目的変数（VRE 結果 学習機作成時に変更？）

| 修正前                | 修正後         |
| ------------------ | ----------- |
| VRE結果 (1:陽性, 2:陰性) | 陽性＝1 / 陰性＝0 |


In [ ]:
# =====================================
# 初期設定
# =====================================
seed = 42              # ランダムシード
sm_k = 5               # SMOTEの近傍数
n_tree = 500           # ランダムフォレストの木の数
test_size = 0.2        # テストデータの割合
pcr_unit_cost=171      # PCR検査の単価（円）
model_types = ["rf", "lr", "elasticnet", "svc", "xgb", "lgb"]  # モデルの種類

# =====================================
# データの前処理
# =====================================
df_nyuin = pd.read_excel(os.path.join(raw_data_path, "日赤VRE 20260203.xlsx"), sheet_name="入院時ASC全例")
df_byoto = pd.read_excel(os.path.join(raw_data_path, "日赤VRE 20260203.xlsx"), sheet_name="病棟ASC全例")


# column名変更
conv_dict = {
    'Pt No': 'patient_id',
    'Sex\n1. Male\n2. Female': 'sex',
    'Age': 'age',
    '入院日': 'admission_date',
    '検査日': 'test_date',
    '入院-検査期間(day)': 'days_from_admission_to_test',
    'Setting\n1. 入院時\n2. 病棟定期': 'sampling_setting',
    'VRE\n1. 陽性\n2. 陰性': 'vre_result',
    '過去のVRE検出歴\n1. あり\n2. なし': 'past_vre_history',
    '感染経路\n1. もちこみ (過去歴なし)\n2. もちこみ（過去歴あり）\n3. 院内新規感染\n4. 該当なし': 'infection_route',
    '入院時申し込み病名': 'admission_diagnosis',
    '過去三ヶ月以内の当院入院歴\n1. あり\n2. なし': 'recent_admission_this_hospital',
    '過去3ヶ月以内他院入院歴\n1. あり\n2. なし': 'recent_admission_other_hospital',
    '入院歴のある病院\n1. 流行地域病院の入院歴あり \n2. 入院歴なし\n3. 非流行地域の入院歴あり': 'hospitalization_region_history',
    '過去3ヶ月以内の介護施設入所歴\n1. あり\n2. なし': 'recent_care_facility_stay',
    '尿Ba\n1. あり\n2. なし': 'urinary_catheter',
    'Pトイレ\n1. あり\n2. なし': 'private_toilet',
    'おむつ\n1. あり\n2. なし': 'diaper_use',
    '過去3ヶ月以内の抗菌薬使用歴\n1. あり\n2. なし': 'recent_antibiotics_use',
    '経管栄養\n1. あり\n2. なし': 'tube_feeding'
}

df_nyuin.rename(columns=conv_dict, inplace=True)
df_nyuin["admission_diagnosis_id"]=df_nyuin["admission_diagnosis"].str.extract(r"^(\d+)\.").astype(int)
df_byoto.rename(columns=conv_dict, inplace=True)



In [ ]:
# admission_diagnosis_id と 元の診断内容の対応表を作成
df_admission_diag_map = (
    df_nyuin.loc[:, ["admission_diagnosis_id", "admission_diagnosis"]]
    .drop_duplicates()
    .sort_values("admission_diagnosis_id")
    .reset_index(drop=True)
)

df_admission_diag_map


In [ ]:
# -----------------------------
# 0/1 二値化する列
# -----------------------------

"""
以下について0(該当なし)を含むレコードを削除, またそのレコードを表示
    "recent_admission_this_hospital"
    "recent_admission_other_hospital"
    "hospitalization_region_history"
    "recent_care_facility_stay"
"""
print("削除された入院データ:")
print(df_nyuin[
    (df_nyuin["recent_admission_this_hospital"] == 0) |
    (df_nyuin["recent_admission_other_hospital"] == 0) |
    (df_nyuin["hospitalization_region_history"] == 0) |
    (df_nyuin["recent_care_facility_stay"] == 0)
])

df_nyuin = df_nyuin[
    (df_nyuin["recent_admission_this_hospital"] != 0) &
    (df_nyuin["recent_admission_other_hospital"] != 0) &
    (df_nyuin["hospitalization_region_history"] != 0) &
    (df_nyuin["recent_care_facility_stay"] != 0)
].copy()

# byotoデータには該当レコードはないため、そのまま

binary_map = {
    "sex": {1: 1, 2: 0},
    "past_vre_history": {1: 1, 2: 0},
    "recent_admission_this_hospital": {1: 1, 0: 0, 2: 0},
    "recent_admission_other_hospital": {1: 1, 0: 0, 2: 0},
    "recent_care_facility_stay": {1: 1, 0: 0, 2: 0},
    "urinary_catheter": {1: 1, 2: 0},
    "private_toilet": {1: 1, 2: 0},
    "diaper_use": {1: 1, 2: 0},
    "recent_antibiotics_use": {1: 1, 2: 0},
    "tube_feeding": {1: 1, 2: 0},
}

def apply_binary_mapping(df):
    for col, mp in binary_map.items():
        if col in df.columns:
            df[col] = df[col].map(mp).astype(int)
    return df


df_nyuin = apply_binary_mapping(df_nyuin)
df_byoto = apply_binary_mapping(df_byoto)


# -----------------------------
# One-Hot 化する列
# -----------------------------
one_hot_cols = [
    "sampling_setting",               # 1:入院時, 2:病棟定期
    "infection_route",                # 1〜4
    "hospitalization_region_history", # 0〜3
]

def apply_onehot(df, columns):
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df = pd.get_dummies(df, columns=[col], prefix=col)
    return df


df_nyuin = apply_onehot(df_nyuin, one_hot_cols)
df_byoto = apply_onehot(df_byoto, one_hot_cols)


# -----------------------------
# admission_diagnosis_id はすでに数値化済み
# -----------------------------
df_nyuin["admission_diagnosis_id"] = df_nyuin["admission_diagnosis_id"].astype(int)

# =====================================
# 特徴量とターゲットの再設定
# =====================================

# 入院データ用の特徴量
feature_columns_nyuin = [
    'sex',
    'age',
    # 'days_from_admission_to_test',
    # 'admission_diagnosis_id',
    'recent_admission_this_hospital',
    'recent_admission_other_hospital',
    # 'hospitalization_region_history_0',
    'hospitalization_region_history_1',
    'hospitalization_region_history_2',
    'hospitalization_region_history_3',
    'recent_care_facility_stay',
    'urinary_catheter',
    'private_toilet',
    'diaper_use',
    'recent_antibiotics_use',
    'tube_feeding'
]

# # sampling_settingとinfection_routeをOne-Hotで追加（存在する場合のみ）
# feature_columns_nyuin += [c for c in df_nyuin.columns if c.startswith("sampling_setting_")]
# feature_columns_nyuin += [c for c in df_nyuin.columns if c.startswith("infection_route_")]

# 病棟データ用（存在する列だけ抽出）
feature_columns_byoto = [
    'sex',
    'age',
    'days_from_admission_to_test',
    'urinary_catheter',
    'private_toilet',
    'diaper_use',
    'recent_antibiotics_use',
    'tube_feeding'
]

# feature_columns_byoto += [c for c in df_byoto.columns if c.startswith("sampling_setting_")]
# feature_columns_byoto += [c for c in df_byoto.columns if c.startswith("infection_route_")]

# =====================================
# X, y の作成
# =====================================

lavel_column = 'vre_result'

X_nyuin = df_nyuin[feature_columns_nyuin].copy()
X_byoto = df_byoto[feature_columns_byoto].copy()

# ターゲット（VRE結果: 陽性1 / 陰性0）
y_nyuin = df_nyuin[lavel_column].apply(lambda x: 1 if x == 1 else 0)
y_byoto = df_byoto[lavel_column].apply(lambda x: 1 if x == 1 else 0)

print("入院データ X_nyuin shape:", X_nyuin.shape)
print("病棟データ X_byoto shape:", X_byoto.shape)

# 陽性率の確認
print("\n入院データ VRE陽性率:", y_nyuin.mean())
print("病棟データ VRE陽性率:", y_byoto.mean())

In [ ]:
# Aのデータフレームを df とする
df_tmp = df_nyuin.copy()

# =========================
# 1. 他院入院歴（地域）を1列に戻す
# =========================
def map_region_history(row):
    if row['hospitalization_region_history_1'] == True:
        return 'VRE流行地域'
    elif row['hospitalization_region_history_3'] == True:
        return 'VRE非流行地域'
    elif row['hospitalization_region_history_2'] == True:
        return '入院歴なし'
    else:
        return '不明'

df_tmp['他院入院歴（地域）'] = df_tmp.apply(map_region_history, axis=1)

# =========================
# 2. VRE結果をラベル化
# =========================
df_tmp['VRE結果'] = df_tmp['vre_result'].map({
    1: 'Positive',
    2: 'Negative'
})

# =========================
# 3. 件数集計
# =========================
count_table = pd.crosstab(df_tmp['他院入院歴（地域）'], df_tmp['VRE結果'])

# 行の順番を指定
order = ['VRE流行地域', 'VRE非流行地域', '入院歴なし']
count_table = count_table.reindex(order)

# Positive / Negative の総数
n_pos = (df_tmp['VRE結果'] == 'Positive').sum()
n_neg = (df_tmp['VRE結果'] == 'Negative').sum()

# =========================
# 4. 割合付きの表を作成
# =========================
result_table = pd.DataFrame(index=count_table.index)

result_table['Positive'] = count_table['Positive'].fillna(0).astype(int).apply(
    lambda x: f"{x} ({x / n_pos * 100:.1f} %)" if n_pos > 0 else f"{x} (0.0 %)"
)

result_table['Negative'] = count_table['Negative'].fillna(0).astype(int).apply(
    lambda x: f"{x} ({x / n_neg * 100:.1f} %)" if n_neg > 0 else f"{x} (0.0 %)"
)

# 表示
result_table

In [ ]:
class_counts = pd.Series(y_nyuin).value_counts(normalize=True)  # normalize=True で割合を出す

print("クラスの割合:")
print(class_counts)
print(f"陽性者:{sum(y_nyuin == 1)}, 陰性者:{sum(y_nyuin == 0)}")

In [ ]:
plot_label_dict = {
    'sex': 'Sex, Male',
    'age': 'Age [IQR]',
    'days_from_admission_to_test': 'Days from admission',
    'vre_result': 'VRE colonization',
    'past_vre_history': 'Prior VRE colonization',
    # 'admission_diagnosis_id': 'Diagnosis ID',

    'recent_admission_this_hospital': 'Prior admission to this hospital',
    'recent_admission_other_hospital': 'Prior admission to another hospital',

    # 'hospitalization_region_history_0': 'Region: none', # この列は存在しない
    'hospitalization_region_history_1': 'Prior admission region: high-prevalence region',
    'hospitalization_region_history_2': 'Prior admission region: no prior admission',
    'hospitalization_region_history_3': 'Prior admission region: low-prevalence region',

    'recent_care_facility_stay': 'Residence in a long-term care facility',
    'urinary_catheter': 'Urinary catheter use',
    'private_toilet': 'Bedside commodes use',
    'diaper_use': 'Diaper use',
    'recent_antibiotics_use': 'Antimicrobial exposure',
    'tube_feeding': 'Enteral feeding',

    'sampling_setting_1': 'Sampling: admission',
    'sampling_setting_2': 'Sampling: ward',

    'infection_route_1': 'Imported (no history)',
    'infection_route_2': 'Imported (with history)',
    'infection_route_3': 'Nosocomial',
    'infection_route_4': 'None',

    'shizuoka_vre_cases_1mo': 'VRE cases (prefecture)',
    'tobu_vre_cases_1mo': 'VRE cases (district)',
    'tobu_vre_screening_negative_1mo': 'Screening –',
    'tobu_vre_screening_positive_1mo': 'Screening +',
    'tobu_vre_screening_rate_1mo': 'VRE Positivity rate',

    'ABHR_consumption_1mo': 'ABHR'}


In [ ]:
def model_comparison_table(results, fig_tag, digits=2):
    rows = []

    metric_map = {
        "Accuracy": ("accuracy_mean", "accuracy_std"),
        "Precision": ("precision_mean", "precision_std"),
        "Recall": ("recall_mean", "recall_std"),
        "F1 Score": ("f1_mean", "f1_std"),
        "ROC AUC": ("auc_mean", "auc_std"),
    }

    for model_type, result in results[fig_tag].items():
        metrics = result["cv_metrics"]

        row = {
            "Tag": fig_tag,
            "Model": model_type,
        }

        for label, (mean_key, std_key) in metric_map.items():
            mean = metrics[mean_key]
            std = metrics[std_key]
            row[label] = f"{mean:.{digits}f} ± {std:.{digits}f}"

        rows.append(row)

    return pd.DataFrame(rows)

# 入院時

In [ ]:
results_nyuin = {}

## 分類機作成

In [ ]:
fig_tag = "nyuin"
results_nyuin[fig_tag] = {}
if not os.path.exists(os.path.join(results_path, fig_tag)):
    os.makedirs(os.path.join(results_path, fig_tag))
label_column = 'vre_result'

X_nyuin = df_nyuin[feature_columns_nyuin].copy()
# ターゲット（VRE結果: 陽性1 / 陰性0）
y_nyuin = df_nyuin[label_column].apply(lambda x: 1 if x == 1 else 0)

for model_type in model_types:
    print(f"=== {model_type.upper()} ROC Curve ===")
    final_model,cv_metrics, final_thr, all_y_true, all_y_proba, figures, tprs, aucs = plot_roc_curve_cv(X_nyuin, y_nyuin, f"{model_type.upper()} CV ROC", seed=seed, model_type=model_type, threshold_method="0.5", plt_show=False)
    figures["roc"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_roc.png"))
    figures["pr"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pr.png"))
    figures["dca_zoom"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_dca_zoom.png"))
    figures["table"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_metrics_table.png"))
    figures["calib"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_calibration.png"))
    figures["pcr"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pcr.png"))
    figures["pcr_target"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pcr_target.png"))
    
    results_nyuin[fig_tag][model_type] = {
        "final_model": final_model,
        "cv_metrics": cv_metrics,
        "final_thr": final_thr,
        "all_y_true": all_y_true,
        "all_y_proba": all_y_proba,
        "figures": figures,
        "tprs": tprs,
        "aucs": aucs
    }
    


comparison_df = model_comparison_table(results_nyuin, fig_tag)
comparison_df.to_csv(
    os.path.join(results_path, fig_tag, "model_comparison.csv"),
    index=False
)

print(comparison_df)

fig_combined = plot_model_roc_comparison(
    results_nyuin[fig_tag],
    labels={
        "rf": "Random Forest",
        "lr": "Logistic Regression",
        "elasticnet": "Elastic Net",
        "svc": "SVC",
        "xgb": "XGBoost",
        "lgb": "LightGBM"
    }
)

fig_combined.savefig(os.path.join(results_path, fig_tag, "model_comparison.png"))


In [ ]:
comparison_df

## 流行情報統合

In [ ]:
df_ryuko = pd.read_excel(os.path.join(raw_data_path, "静岡県・東部保健所管内のデータ.xlsx"), sheet_name="静岡県感染者数")
df_ryuko2 = pd.read_excel(os.path.join(raw_data_path, "静岡県・東部保健所管内のデータ.xlsx"), sheet_name="東部保健所管内スクリーニング件数・新規スクリーニング陽性数")

# Date列をkeyとして結合
df_ryuko = pd.merge(df_ryuko, df_ryuko2, on="Date", how="outer")

# column名変更
conv_dict = {
    '静岡県全VRE感染者数(人)': 'shizuoka_vre_cases',
    '東部保健所管内VRE感染者数（人）': 'tobu_vre_cases',
    'VREスクリーニング陰性数（人）': 'tobu_vre_screening_negative',
    'VREスクリーニング新規陽性数（人）': 'tobu_vre_screening_positive',
}

df_ryuko.rename(columns=conv_dict, inplace=True)
df_ryuko["tobu_vre_screening_rate"] = df_ryuko["tobu_vre_screening_positive"] / (df_ryuko["tobu_vre_screening_negative"] + df_ryuko["tobu_vre_screening_positive"])

In [ ]:
# --- 1. 入院日(admission_date)をdatetime型にして月初に丸める ---
df_nyuin['admission_date'] = pd.to_datetime(df_nyuin['admission_date'])
df_nyuin['month_start'] = df_nyuin['admission_date'].values.astype('datetime64[M]')
df_nyuin['month_start'] = pd.to_datetime(df_nyuin['month_start'])  # Timestamp型に戻す

# --- 2. df_ryukoの日付(Date)もdatetime型にし月初に丸める ---
df_ryuko['Date'] = pd.to_datetime(df_ryuko['Date'])
df_ryuko['Date'] = df_ryuko['Date'].values.astype('datetime64[M]')
df_ryuko['Date'] = pd.to_datetime(df_ryuko['Date'])

# --- 3. df_nyuinに1ヶ月前、2ヶ月前の月初日列を作る ---
df_nyuin['last_month_start'] = df_nyuin['month_start'] - pd.DateOffset(months=1)
df_nyuin['two_months_ago_start'] = df_nyuin['month_start'] - pd.DateOffset(months=2)

# --- 4. df_ryukoを1ヶ月前・2ヶ月前用にコピーし、suffixで列名を分ける ---
df_ryuko_1mo = df_ryuko.add_suffix('_1mo').rename(columns={'Date_1mo': 'Date_1mo'})
df_ryuko_2mo = df_ryuko.add_suffix('_2mo').rename(columns={'Date_2mo': 'Date_2mo'})

# --- 5. 1ヶ月前分をマージ ---
df_merged = pd.merge(df_nyuin, df_ryuko_1mo,
                     left_on='last_month_start', right_on='Date_1mo',
                     how='left')

# --- 6. の処理は最後にまとめるため削除 ---
# cols_1mo = [col for col in df_merged.columns if col.endswith('_1mo')]
# df_merged.drop(columns=cols_1mo, inplace=True)

# --- 7. 2ヶ月前分をマージ ---
df_merged = pd.merge(df_merged, df_ryuko_2mo,
                     left_on='two_months_ago_start', right_on='Date_2mo',
                     how='left')

# --- 9. 結果を元の変数に代入 ---
df_nyuin = df_merged

# 結果の確認
# print(df_nyuin.columns)

In [ ]:
features_1mo = [
    'shizuoka_vre_cases_1mo', 
    'tobu_vre_cases_1mo', 
    # 'tobu_vre_screening_negative_1mo', 
    # 'tobu_vre_screening_positive_1mo',
    'tobu_vre_screening_rate_1mo'
]

features_2mo = [
    'shizuoka_vre_cases_2mo', 
    'tobu_vre_cases_2mo', 
    # 'tobu_vre_screening_negative_2mo', 
    # 'tobu_vre_screening_positive_2mo',
    'tobu_vre_screening_rate_2mo'
]

df_nyuin_nan_removed_1mo = df_nyuin.dropna(subset=features_1mo)
# 削除した列を表示
print(f"removed {df_nyuin.shape[0] - df_nyuin_nan_removed_1mo.shape[0]} samples due to NaN in 1mo features")

df_nyuin_nan_removed_2mo = df_nyuin.dropna(subset=features_1mo + features_2mo)
print(f"After merging, dataset size: {df_nyuin_nan_removed_2mo.shape[0]} samples")
# 削除した列を表示
print(f"removed {df_nyuin.shape[0] - df_nyuin_nan_removed_2mo.shape[0]} samples due to NaN in 1mo + 2mo features")

fig_tag = "nyuin_1mo"
results_nyuin[fig_tag] = {}
if not os.path.exists(os.path.join(results_path, fig_tag)):
    os.makedirs(os.path.join(results_path, fig_tag))

X_nyuin_nan_removed_1mo = df_nyuin_nan_removed_1mo[feature_columns_nyuin + features_1mo].copy()
y_nyuin_nan_removed_1mo = df_nyuin_nan_removed_1mo[lavel_column].apply(lambda x: 1 if x == 1 else 0)

for model_type in model_types:
    print(f"=== {model_type.upper()} CV ROC ===")
    final_model,cv_metrics, final_thr, all_y_true, all_y_proba, figures, tprs, aucs = plot_roc_curve_cv(X_nyuin_nan_removed_1mo, y_nyuin_nan_removed_1mo, f"{model_type.upper()} CV ROC + 1mo", seed=seed, model_type=model_type, threshold_method="0.5", plt_show=False)
    figures["roc"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_roc.png"))
    figures["pr"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pr.png"))
    figures["dca_zoom"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_dca_zoom.png"))
    figures["table"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_metrics_table.png"))
    figures["calib"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_calibration.png"))
    figures["pcr"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pcr.png"))
    figures["pcr_target"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pcr_target.png"))
    
    results_nyuin[fig_tag][model_type] = {
        "final_model": final_model,
        "cv_metrics": cv_metrics,
        "final_thr": final_thr,
        "all_y_true": all_y_true,
        "all_y_proba": all_y_proba,
        "figures": figures,
        "tprs": tprs,
        "aucs": aucs
    }


comparison_df = model_comparison_table(results_nyuin, fig_tag)

comparison_df.to_csv(
    os.path.join(results_path, fig_tag, "model_comparison.csv"),
    index=False
)

print(comparison_df)

fig_combined = plot_model_roc_comparison(
    results_nyuin[fig_tag],
    labels={
        "rf": "Random Forest",
        "lr": "Logistic Regression",
        "elasticnet": "Elastic Net",
        "svc": "SVC",
        "xgb": "XGBoost",
        "lgb": "LightGBM"
    }
)

fig_combined.savefig(os.path.join(results_path, fig_tag, "model_comparison.png"))


# 病棟

In [ ]:
results_byoto = {}

In [ ]:
fig_tag = "byoto"
results_byoto[fig_tag] = {}
if not os.path.exists(os.path.join(results_path, fig_tag)):
    os.makedirs(os.path.join(results_path, fig_tag))
for model_type in model_types:
    print(f"=== {model_type.upper()} CV ROC (Byoto) ===")
    final_model,cv_metrics, final_thr, all_y_true, all_y_proba, figures, tprs, aucs  = plot_roc_curve_cv(X_byoto, y_byoto, f"{model_type.upper()} CV ROC", seed=seed, model_type=model_type, threshold_method="0.5", plt_show=False)
    figures["roc"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_roc.png"))
    figures["pr"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pr.png"))
    figures["dca_zoom"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_dca_zoom.png"))
    figures["table"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_metrics_table.png"))
    figures["calib"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_calibration.png"))
    figures["pcr"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pcr.png"))
    figures["pcr_target"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pcr_target.png"))
    
    results_byoto[fig_tag][model_type] = {
        "final_model": final_model,
        "cv_metrics": cv_metrics,
        "final_thr": final_thr,
        "all_y_true": all_y_true,
        "all_y_proba": all_y_proba,
        "figures": figures,
        "tprs": tprs,
        "aucs": aucs
    }

comparison_df = model_comparison_table(results_byoto, fig_tag)
comparison_df.to_csv(
    os.path.join(results_path, fig_tag, "model_comparison.csv"),
    index=False
)

print(comparison_df)

fig_combined = plot_model_roc_comparison(
    results_byoto[fig_tag],
    labels={
        "rf": "Random Forest",
        "lr": "Logistic Regression",
        "elasticnet": "Elastic Net",
        "svc": "SVC",
        "xgb": "XGBoost",
        "lgb": "LightGBM"
    }
)

fig_combined.savefig(os.path.join(results_path, fig_tag, "model_comparison.png"))


## アルコール使用状況統合

In [ ]:
df_ABHR = pd.read_excel(os.path.join(raw_data_path, "日赤アルコール使用量.xlsx"), sheet_name="Sheet1")

# column名変更
conv_dict = {
    '月': 'Date',
    'アルコール消費量（mL/患者/day）': 'ABHR_consumption',
}

df_ABHR.rename(columns=conv_dict, inplace=True)


In [ ]:

# test_dateをdatetime型に変換し、月初日に変換
df_byoto['test_date'] = pd.to_datetime(df_byoto['test_date'])
df_byoto['month_start'] = df_byoto['test_date'].values.astype('datetime64[M]')

# df_ABHRの日付もdatetimeにしておく
df_ABHR['Date'] = pd.to_datetime(df_ABHR['Date'])

# 月初でマージ
df_merged = pd.merge(df_byoto, df_ABHR, left_on='month_start', right_on='Date', how='left')

# 不要な列を整理（必要なら）
df_merged.drop(columns=['month_start', 'Date'], inplace=True)

df_byoto = df_merged

In [ ]:

# --- 1. 入院日(test_date)をdatetime型にして月初に丸める ---
df_byoto['test_date'] = pd.to_datetime(df_byoto['test_date'])
df_byoto['month_start'] = df_byoto['test_date'].values.astype('datetime64[M]')
df_byoto['month_start'] = pd.to_datetime(df_byoto['month_start'])  # Timestamp型に戻す

# --- 2. df_ABHRの日付(Date)もdatetime型にし月初に丸める ---
df_ABHR['Date'] = pd.to_datetime(df_ABHR['Date'])
df_ABHR['Date'] = df_ABHR['Date'].values.astype('datetime64[M]')
df_ABHR['Date'] = pd.to_datetime(df_ABHR['Date'])

# --- 3. df_byotoに1ヶ月前、2ヶ月前の月初日列を作る ---
df_byoto['last_month_start'] = df_byoto['month_start'] - pd.DateOffset(months=1)
df_byoto['two_months_ago_start'] = df_byoto['month_start'] - pd.DateOffset(months=2)

# --- 4. df_ABHRを1ヶ月前・2ヶ月前用にコピーし、suffixで列名を分ける ---
df_ABHR_current = df_ABHR.add_suffix('_current').rename(columns={'Date': 'Date'})
df_ABHR_1mo = df_ABHR.add_suffix('_1mo').rename(columns={'Date_1mo': 'Date_1mo'})
df_ABHR_2mo = df_ABHR.add_suffix('_2mo').rename(columns={'Date_2mo': 'Date_2mo'})

# --- 5. 1ヶ月前分をマージ ---
df_merged = pd.merge(df_byoto, df_ABHR_current,
                     left_on='month_start', right_on='Date_current',
                     how='left')

df_merged = pd.merge(df_merged, df_ABHR_1mo,
                     left_on='last_month_start', right_on='Date_1mo',
                     how='left')

df_merged = pd.merge(df_merged, df_ABHR_2mo,
                     left_on='two_months_ago_start', right_on='Date_2mo',
                     how='left')

# --- 9. 結果を元の変数に代入 ---
df_byoto = df_merged

# 結果の確認
print(df_merged.columns)



In [ ]:
# 陽性者数、陰性者数を確認
positive_count = df_byoto[df_byoto['vre_result'] == 1].shape[0]
negative_count = df_byoto[df_byoto['vre_result'] == 2].shape[0]
print(f"陽性者数: {positive_count}, 陰性者数: {negative_count}")

In [ ]:
# 特徴量にアルコール使用状況を追加
X_byoto = df_byoto[feature_columns_byoto + ['ABHR_consumption_1mo']]

fig_tag = "byoto_ABHR_1mo"
results_byoto[fig_tag] = {}
if not os.path.exists(os.path.join(results_path, fig_tag)):
    os.makedirs(os.path.join(results_path, fig_tag))
for model_type in model_types:
    print(f"=== {model_type.upper()} CV ROC ===")
    final_model,cv_metrics, final_thr, all_y_true, all_y_proba, figures, tprs, aucs  = plot_roc_curve_cv(X_byoto, y_byoto, f"{model_type.upper()} CV ROC + 1mo", seed=seed, model_type=model_type, threshold_method="0.5", plt_show=False)
    figures["roc"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_roc.png"))
    figures["pr"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pr.png"))
    figures["dca_zoom"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_dca_zoom.png"))
    figures["table"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_metrics_table.png"))
    figures["calib"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_calibration.png"))
    figures["pcr"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pcr.png"))
    figures["pcr_target"].savefig(os.path.join(results_path, fig_tag, f"{model_type}_pcr_target.png"))
    
    
    results_byoto[fig_tag][model_type] = {
        "final_model": final_model,
        "cv_metrics": cv_metrics,
        "final_thr": final_thr,
        "all_y_true": all_y_true,
        "all_y_proba": all_y_proba,
        "figures": figures,
        "tprs": tprs,
        "aucs": aucs
    }
    
comparison_df = model_comparison_table(results_byoto, fig_tag)
comparison_df.to_csv(
    os.path.join(results_path, fig_tag, "model_comparison.csv"),
    index=False
)

print(comparison_df)

fig_combined = plot_model_roc_comparison(
    results_byoto[fig_tag],
    labels={
        "rf": "Random Forest",
        "lr": "Logistic Regression",
        "elasticnet": "Elastic Net",
        "svc": "SVC",
        "xgb": "XGBoost",
        "lgb": "LightGBM"
    }
)

fig_combined.savefig(os.path.join(results_path, fig_tag, "model_comparison.png"))
